In [ ]:
# 实现短期记忆
# langchain将会话历史存在AgentState中
# 每次对话生成一次AgentStateSnapshot,组合在一起就是完整对话记忆

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
# langchain提供的checkpointer的默认实现，基于内存存储
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

# 1.加载环境变量
from dotenv import load_dotenv
load_dotenv()

# 2.初始化模型
model = init_chat_model(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}},
)

# 3.定义工具

# 4.创建Agent,绑定模型和工具,checkpointer
agent =create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver()# 所有的AgentStateSnapshot都是通过Checkpointer对象来保存的
)
# 有多种存储方式
# - InMemorySaver：基于内存存储
# - SqliteSaver：基于sqlite数据库存储
# - PostgreSaver：基于PostgreSql数据存储

# 5.调用agent
config = {"configurable": {"thread_id": "thread_01"}}# 为了区分不同的会话记忆，不同会话需要设定各自的thread_id

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我叫科比，我最喜欢猫猫。")]},
    config # 调用时添加thread_id，区分不同会话 # type: ignore
)

print(response)



{'messages': [HumanMessage(content='你好，我叫虎哥，我最喜欢猫猫。', additional_kwargs={}, response_metadata={}, id='4118676c-c385-4314-b8b9-32918c39ab8b'), AIMessage(content='虎哥你好呀！猫猫确实超可爱，毛茸茸又傲娇的样子谁顶得住～ 你是喜欢橘猫、狸花猫，还是那种软乎乎的布偶猫？有没有养猫，或者只是云吸猫爱好者？🐱', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 15, 'total_tokens': 68, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 15}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '34669423-e86b-42e1-b32d-b96a7d21d4fe', 'finish_reason': 'stop', 'logprobs': None, 'normalized_finish_reason': 'stop'}, id='lc_run--01a2136a-52c9-76fc-9162-8e7e81927700-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 

In [ ]:
# 第二次调用，询问我的信息，这次带上thread_id，唤起记忆
response = agent.invoke(
    {"messages": [HumanMessage(content="我最喜欢的动物是什么？")]},
    config # 调用时添加thread_id # type: ignore
)

print(response)

{'messages': [HumanMessage(content='你好，我叫虎哥，我最喜欢猫猫。', additional_kwargs={}, response_metadata={}, id='4118676c-c385-4314-b8b9-32918c39ab8b'), AIMessage(content='虎哥你好呀！猫猫确实超可爱，毛茸茸又傲娇的样子谁顶得住～ 你是喜欢橘猫、狸花猫，还是那种软乎乎的布偶猫？有没有养猫，或者只是云吸猫爱好者？🐱', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 15, 'total_tokens': 68, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 15}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '34669423-e86b-42e1-b32d-b96a7d21d4fe', 'finish_reason': 'stop', 'logprobs': None, 'normalized_finish_reason': 'stop'}, id='lc_run--01a2136a-52c9-76fc-9162-8e7e81927700-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 

: 